### Coding from scratch ###

In [5]:
## The first thing that is there in the RAG pipeline is that we must have some documents from where we can take the info from

## We use this document break down into chunks and convert into vector embeddings

## Storing into a vector database part comes after all of this 


import numpy as np
import pandas as pd
import chromadb
from langchain_community import PDFLoader, 

SyntaxError: trailing comma not allowed without surrounding parentheses (354313659.py, line 11)

In [6]:
documents = [
    "RAG is full form of retreival augmented generation",
    "LLM is full form for Large Language Model",
    "Transformer architecture is the heart of the modern LLM ecosystem"
]

#Imagine these as the documents that we have

embeddings = [
    [0.1 , 0.2, 0.3],
    [0.2 , 0.2, 0.4],
    [0.4, 0.3 , 0.1]
]

print("Number of documents", len(documents))
print("Number of embeddings", len(embeddings))




Number of documents 3
Number of embeddings 3


In [7]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(name = "test_collection")


## Add the documents and embeddings into this collection 

collection.add(ids= ["id1" , "id2", "id3"],
               embeddings= embeddings,
               documents= documents)

collection.count()

collection.get(ids=["id1"])

{'ids': ['id1'],
 'embeddings': None,
 'documents': ['RAG is full form of retreival augmented generation'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None]}

In [8]:
## Do similarity search on the db based on an example query

query_embedding = [0.11 , 0.21 , 0.29]


collection.query(
    query_embeddings= [query_embedding],
    n_results=2
)

{'ids': [['id1', 'id2']],
 'embeddings': None,
 'documents': [['RAG is full form of retreival augmented generation',
   'LLM is full form for Large Language Model']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.00030000018887221813, 0.02030000276863575]]}

In [9]:
results = collection.query(
    query_embeddings=[[0.11, 0.21, 0.29]],
    n_results=3
)

print("IDs:", results["ids"])
print("Documents:", results["documents"])
print("Distances:", results["distances"])

IDs: [['id1', 'id2', 'id3']]
Documents: [['RAG is full form of retreival augmented generation', 'LLM is full form for Large Language Model', 'Transformer architecture is the heart of the modern LLM ecosystem']]
Distances: [[0.00030000018887221813, 0.02030000276863575, 0.1283000111579895]]


In [10]:
import sentence_transformers

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print(type(embeddings))
print(embeddings.shape)

/Users/ishanchowdhury/rag-learning-path/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7397.36it/s]


<class 'numpy.ndarray'>
(3, 384)


In [11]:
### Add the actual embeddings instead of the fake ones into the collection 

collection = client.get_or_create_collection(
    name="real_embeddings_collection"
)

collection.add(ids = ["id1" , "id2", "id3"],
               embeddings= embeddings,
               documents= documents)

print("Number of records:", collection.count())

Number of records: 3


In [12]:
query = "What is RAG?"

query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=1
)

print("ID:", results["ids"])
print("Document:", results["documents"])
print("Distance:", results["distances"])

ID: [['id1']]
Document: [['RAG is full form of retreival augmented generation']]
Distance: [[0.7606343030929565]]


In [13]:
query = "Tell me about RAG and LLMs"

result = model.encode(query)

#This is the embedding for this string

collection.query(query_embeddings= [result],
                 n_results= 2)

print("IDs:", results["ids"])
print("Documents:", results["documents"])
print("Distances:", results["distances"])


IDs: [['id1']]
Documents: [['RAG is full form of retreival augmented generation']]
Distances: [[0.7606343030929565]]


In [14]:
retrieved_documents = results["documents"][0]

for doc in retrieved_documents:
    print(doc)

RAG is full form of retreival augmented generation


In [15]:
retrieved_documents = results["documents"][0]
context = "\n\n".join(retrieved_documents)
print(context)

RAG is full form of retreival augmented generation


In [16]:
prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""


print(prompt)


Answer the question using only the provided context.

Context:
RAG is full form of retreival augmented generation

Question:
Tell me about RAG and LLMs

Answer:



In [17]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

outputs = llm.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer:", answer)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4073.95it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Answer: RAG is full form of retreival augmented generation


In [18]:
### We did not include chunking before 

document = """
Retrieval Augmented Generation allows a language model to retrieve
relevant information from an external knowledge base before generating
an answer. This reduces the need for the model to rely only on its
parametric knowledge. RAG is commonly used with vector databases,
embedding models, and large language models.
"""

words = document.split()

print(words)
print("Number of words:", len(words))

['Retrieval', 'Augmented', 'Generation', 'allows', 'a', 'language', 'model', 'to', 'retrieve', 'relevant', 'information', 'from', 'an', 'external', 'knowledge', 'base', 'before', 'generating', 'an', 'answer.', 'This', 'reduces', 'the', 'need', 'for', 'the', 'model', 'to', 'rely', 'only', 'on', 'its', 'parametric', 'knowledge.', 'RAG', 'is', 'commonly', 'used', 'with', 'vector', 'databases,', 'embedding', 'models,', 'and', 'large', 'language', 'models.']
Number of words: 47


In [ ]:
def chunk_text(text, chunk_size):
    words = text.split()
    
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [23]:
document = """
Retrieval Augmented Generation allows a language model to retrieve
relevant information from an external knowledge base before generating
an answer. This reduces the need for the model to rely only on its
parametric knowledge. RAG is commonly used with vector databases,
embedding models, and large language models.
"""


chunks = chunk_text(document, 20)

chunks

['Retrieval Augmented Generation allows a language model to retrieve relevant information from an external knowledge base before generating an answer.',
 'This reduces the need for the model to rely only on its parametric knowledge. RAG is commonly used with vector',
 'databases, embedding models, and large language models.']